# 🎓 Student Dropout — Preprocessing & Feature Engineering

**Phase 2** of the Student Dropout Prediction pipeline.

**Input:** `../data/raw/dataset.csv`  
**Output:** `../data/processed/processed_dataset.csv`

**Steps:**
1. Data Loading
2. Target Encoding (multi-class + binary)
3. Feature Engineering (EDA-validated hypotheses)
4. Safe Feature Scaling (scaled suffix, originals preserved for SHAP)
5. Export

## 1. Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/raw/dataset.csv', sep=None, engine='python')
print('Initial shape:', df.shape)
display(df.head())

Initial shape: (4424, 35)


,﻿Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 2. Target Encoding

- `Target_Encoded`: multi-class label — Graduate=0, Dropout=1, Enrolled=2
- `Target_Binary`: binary label for dropout modelling — Dropout=1, Graduate=0, Enrolled=NaN (drop during binary training)

In [2]:
target_multiclass_map = {'Graduate': 0, 'Dropout': 1, 'Enrolled': 2}
df['Target_Encoded'] = df['Target'].map(target_multiclass_map)

target_binary_map = {'Dropout': 1, 'Graduate': 0, 'Enrolled': np.nan}
df['Target_Binary'] = df['Target'].map(target_binary_map)

print('Target_Encoded distribution:')
print(df['Target_Encoded'].value_counts())
print('\nTarget_Binary distribution (NaN = Enrolled, excluded from binary training):')
print(df['Target_Binary'].value_counts(dropna=False))

Target_Encoded distribution:
Target_Encoded
0    2209
1    1421
2     794
Name: count, dtype: int64

Target_Binary distribution (NaN = Enrolled, excluded from binary training):
Target_Binary
0.0    2209
1.0    1421
NaN     794
Name: count, dtype: int64


## 3. Feature Engineering

EDA-validated risk indicators:

| Feature | Description |
|---|---|
| `zero_approved_units_flag` | 1 if 0 approved units in both semesters (Hypothesis 1) |
| `financial_stress_flag` | 1 if Debtor AND tuition not up to date (Hypothesis 2) |
| `semester_performance_delta` | 2nd sem approved − 1st sem approved |
| `engagement_ratio_1st_sem` | evaluations / (enrolled + 0.001) for sem 1 |
| `engagement_ratio_2nd_sem` | evaluations / (enrolled + 0.001) for sem 2 |

In [3]:
# Hypothesis 1: zero approved units in both semesters
df['zero_approved_units_flag'] = (
    (df['Curricular units 1st sem (approved)'] == 0) &
    (df['Curricular units 2nd sem (approved)'] == 0)
).astype(int)

# Hypothesis 2: financial stress (debtor + tuition not up to date)
df['financial_stress_flag'] = (
    (df['Debtor'] == 1) &
    (df['Tuition fees up to date'] == 0)
).astype(int)

# Academic trajectory
df['semester_performance_delta'] = (
    df['Curricular units 2nd sem (approved)'] -
    df['Curricular units 1st sem (approved)']
)

# Engagement ratios
df['engagement_ratio_1st_sem'] = (
    df['Curricular units 1st sem (evaluations)'] /
    (df['Curricular units 1st sem (enrolled)'] + 0.001)
)
df['engagement_ratio_2nd_sem'] = (
    df['Curricular units 2nd sem (evaluations)'] /
    (df['Curricular units 2nd sem (enrolled)'] + 0.001)
)

print('Engineered features added:')
eng_cols = [
    'zero_approved_units_flag', 'financial_stress_flag',
    'semester_performance_delta', 'engagement_ratio_1st_sem', 'engagement_ratio_2nd_sem'
]
display(df[eng_cols].describe())

Engineered features added:


,zero_approved_units_flag,financial_stress_flag,semester_performance_delta,engagement_ratio_1st_sem,engagement_ratio_2nd_sem
count,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000
mean,0.152803,0.055606,-0.270796,1.290125,1.260882
std,0.359838,0.229185,1.340645,0.567216,0.579296
min,0.000000,0.000000,-9.000000,0.000000,0.000000
25%,0.000000,0.000000,-1.000000,0.999857,0.999833
50%,0.000000,0.000000,0.000000,1.199760,1.166472
75%,0.000000,0.000000,0.000000,1.599680,1.599680
max,1.000000,1.000000,6.000000,3.499417,3.832695


## 4. Safe Feature Scaling

Continuous features are scaled using `StandardScaler` and saved with a `_scaled` suffix.  
**Original columns are preserved** for SHAP explainability in Phase 5.

In [4]:
SCALE_COLS = [
    'Age at enrollment',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Unemployment rate',
    'Inflation rate',
    'GDP',
    'semester_performance_delta'
]

scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[SCALE_COLS])
scaled_col_names = [f'{c}_scaled' for c in SCALE_COLS]

df[scaled_col_names] = scaled_values

print('Scaled columns added (originals preserved):')
display(df[scaled_col_names].describe().round(4))

Scaled columns added (originals preserved):


,Age at enrollment_scaled,Curricular units 1st sem (grade)_scaled,Curricular units 2nd sem (grade)_scaled,Unemployment rate_scaled,Inflation rate_scaled,GDP_scaled,semester_performance_delta_scaled
count,4424.0000,4424.0000,4424.0000,4424.0000,4424.0000,4424.0000,4424.0000
mean,-0.0000,0.0000,-0.0000,-0.0000,0.0000,0.0000,-0.0000
std,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001
min,-0.8258,-2.1971,-1.9635,-1.4890,-1.4669,-1.7897,-6.5119
25%,-0.5622,0.0742,0.0998,-0.8133,-0.6712,-0.7499,-0.5440
50%,-0.4304,0.3396,0.3781,-0.1750,0.1244,0.1401,0.2020
75%,0.2287,0.5697,0.5956,0.8762,0.9923,0.7878,0.2020
max,6.1599,1.7002,1.6009,1.7397,1.7880,1.5456,4.6780


## 5. Export

In [6]:
output_path = '../data/processed/processed_dataset.csv'
df.to_csv(output_path, index=False)

print(f'Saved to: {output_path}')
print(f'Final shape: {df.shape}')
df.head()

Saved to: ../data/processed/processed_dataset.csv
Final shape: (4424, 49)


,﻿Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,semester_performance_delta,engagement_ratio_1st_sem,engagement_ratio_2nd_sem,Age at enrollment_scaled,Curricular units 1st sem (grade)_scaled,Curricular units 2nd sem (grade)_scaled,Unemployment rate_scaled,Inflation rate_scaled,GDP_scaled,semester_performance_delta_scaled
0,1,8,5,2,1,1,1,13,10,6,...,0,0.000000,0.000000,-0.430363,-2.197102,-1.963489,-0.287638,0.124386,0.765761,0.202012
1,1,6,1,11,1,1,1,1,3,4,...,0,0.999833,0.999833,-0.562168,0.693599,0.659562,0.876222,-1.105222,0.347199,0.202012
2,1,1,5,5,1,1,1,22,27,10,...,0,0.000000,0.000000,-0.562168,-2.197102,-1.963489,-0.287638,0.124386,0.765761,0.202012
3,1,8,2,15,1,1,1,23,27,6,...,-1,1.333111,1.666389,-0.430363,0.575611,0.416450,-0.813253,-1.466871,-1.375511,-0.543982
4,2,12,1,3,0,1,1,22,28,10,...,1,1.499750,0.999833,2.864765,0.349468,0.531608,0.876222,-1.105222,0.347199,0.948006
